In [1]:
import sys
import numpy as np
import pandas as pd
import sklearn

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)


Python: 3.12.2 (v3.12.2:6abddd9f6a, Feb  6 2024, 17:02:06) [Clang 13.0.0 (clang-1300.0.29.30)]
NumPy: 2.0.2
pandas: 2.3.3
scikit-learn: 1.6.1


# Step 2 — Load and Inspect

Student ID seed: 1399


In [6]:
raw = pd.read_csv("data/almaty_apartments_raw.csv")

ids = raw["listing_id"].drop_duplicates().sample(
    frac=0.9,
    random_state=1399
)

df = raw[raw["listing_id"].isin(ids)].reset_index(drop=True)

print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df.describe(include="all").T)

(1156, 11)
listing_id           object
listed_date          object
district             object
rooms                object
area_m2              object
floor                 int64
total_floors          int64
year_built          float64
building_type        object
ceiling_height_m    float64
price_kzt            object
dtype: object
listing_id            0
listed_date           0
district              0
rooms                 0
area_m2               0
floor                 0
total_floors          0
year_built           54
building_type         0
ceiling_height_m    133
price_kzt             0
dtype: int64
                   count unique         top freq         mean        std  min  \
listing_id          1156   1080   ALM-11147    3          NaN        NaN  NaN   
listed_date         1156    245  2026-04-21   11          NaN        NaN  NaN   
district            1156     48       Medeu  147          NaN        NaN  NaN   
rooms               1156      6           2  435          NaN     

In [7]:
print("listed_date:")
print(df["listed_date"].sample(10, random_state=1399).tolist())

print("\nrooms:")
print(df["rooms"].sample(10, random_state=1399).tolist())

print("\narea_m2:")
print(df["area_m2"].sample(10, random_state=1399).tolist())

print("\nprice_kzt:")
print(df["price_kzt"].sample(10, random_state=1399).tolist())


listed_date:
['2026-06-06', '2026-05-05', '2026-07-09', '2026-08-30', '2026-01-02', '2026-02-25', '2026-06-02', '2026-01-11', '2026-03-28', '2026-08-23']

rooms:
['2', 'studio', '2', '3', '2', '1', '1', '4', '2', '2']

area_m2:
['57.8', '29.9', '59.3', '78.1', '63.6', '34.3', '50.2', '106.5', '55.8', '51.1']

price_kzt:
['60870000', '16260000', '48780000', '84310000', '44300000', '17760000', '36930000', '88320000', '45840000', '49800000']


In [8]:
for col in ["rooms", "area_m2", "price_kzt"]:
    numeric = pd.to_numeric(df[col], errors="coerce")
    bad_values = df.loc[numeric.isna(), col].value_counts()

    print(f"\n--- {col}: non-numeric values ---")
    print(bad_values)
    


--- rooms: non-numeric values ---
rooms
studio    31
Name: count, dtype: int64

--- area_m2: non-numeric values ---
area_m2
46.8 m²     2
69,2        2
62,2        2
61.1 m²     2
35.4 m²     2
           ..
37,3        1
120,1       1
50.0 m²     1
66.3 m²     1
114.9 m²    1
Name: count, Length: 84, dtype: int64

--- price_kzt: non-numeric values ---
price_kzt
82 080 000 ₸    2
31,620,000      2
81 730 000 ₸    2
44 880 000 ₸    2
30,730,000      1
               ..
19,530,000      1
46 050 000 ₸    1
56 650 000 ₸    1
40 150 000 ₸    1
105,060,000     1
Name: count, Length: 88, dtype: int64


### Q1 — Wrong data types

 `listed_date` has type `object`, but it represents dates and should be converted to `datetime`.
 `rooms` has type `object` because it contains numeric values stored as text and the value `studio` (31 rows). It should be numeric after converting `studio` to 1.
 `area_m2` has type `object` because some values contain comma decimal separators and units such as `m²` (for example, `46,8 m²` and `61.1 m²`). It should be numeric.
 `price_kzt` has type `object` because some prices contain spaces, commas, and the `₸` symbol (for example, `82 080 000 ₸` and `31,620,000`). It should be numeric.

The other `object` columns such as `listing_id`, `district`, and `building_type` are categorical/text variables, so their current type is reasonable at this stage.

In [9]:
exact_duplicates = df.duplicated().sum()

repost_rows = df.duplicated(subset="listing_id", keep=False).sum()
repost_ids = df.loc[
    df.duplicated(subset="listing_id", keep=False),
    "listing_id"
].nunique()

print("Exact duplicate rows:", exact_duplicates)
print("Rows belonging to reposted listing IDs:", repost_rows)
print("Unique listing IDs with reposts:", repost_ids)

Exact duplicate rows: 33
Rows belonging to reposted listing IDs: 149
Unique listing IDs with reposts: 73


In [10]:
reposts = df[
    df.duplicated(subset="listing_id", keep=False)
].sort_values(["listing_id", "listed_date"])

reposts[
    ["listing_id", "listed_date", "district", "rooms", "area_m2", "price_kzt"]
].head(20)

,listing_id,listed_date,district,rooms,area_m2,price_kzt
967,ALM-10003,2026-05-04,Auezov,2,48.6,28320000
6,ALM-10003,2026-05-29,Auezov,2,48.6,26120000
816,ALM-10003,2026-05-29,Auezov,2,48.6,26120000
937,ALM-10009,2025-12-03,Medeuskiy,1,33.4,42700000
170,ALM-10009,2026-01-19,Medeuskiy,1,33.4,39400000
460,ALM-10016,2026-01-20,Auezov,1,39.7,35130000
438,ALM-10016,2026-01-29,Auezov,1,39.7,32100000
241,ALM-10021,2026-04-06,Alatau,3,71.3,32300000
596,ALM-10021,2026-05-09,Alatau,3,71.3,30090000
47,ALM-10056,2026-02-08,Bostandyk,4,107.2,101060000


### Q2 — Duplicates, reposts, and grain

I found **33 exact duplicate rows**.
 **149 rows** belong to listing IDs that appear more than once.
 There are **73 unique listing IDs with repeated records**.
 Exact duplicates and reposts are not the same problem. An exact duplicate has identical values in all columns, while a repost has the same `listing_id` but may have a different `listed_date`, price, or other values.
 These groups can overlap, so 33 and 149 should not simply be added together.

**Intended grain after cleaning:** one row represents one unique apartment listing (`listing_id`) after exact duplicates are removed and reposts are resolved.

In [12]:
print("floor:", df["floor"].min(), "to", df["floor"].max())
print("total_floors:", df["total_floors"].min(), "to", df["total_floors"].max())
print("year_built:", df["year_built"].min(), "to", df["year_built"].max())
print("ceiling_height_m:", df["ceiling_height_m"].min(), "to", df["ceiling_height_m"].max())

print("\nDistrict values:")
print(df["district"].value_counts())

print("\nBuilding types:")
print(df["building_type"].value_counts())
print("Rows where floor > total_floors:")
print((df["floor"] > df["total_floors"]).sum())

floor: -1 to 25
total_floors: 3 to 25
year_built: 0.0 to 2024.0
ceiling_height_m: 0.0 to 3.2

District values:
district
Medeu            147
Bostandyk        134
Turksib          129
Auezov           124
Zhetysu          122
Almaly           122
Alatau           103
Nauryzbay        102
Наурызбайский     10
medeu             10
turksib            9
AUEZOV             7
 Auezov            7
bostandyk          7
 Zhetysu           6
Nauryzbayskiy      6
auezov             6
Auezovskiy         6
TURKSIB            5
Bostandykskiy      5
nauryzbay          5
 Turksib           5
BOSTANDYK          5
Alatauskiy         5
Almalinskiy        4
ZHETYSU            4
 Alatau            4
ALMALY             4
Медеуский          4
 Bostandyk         4
almaly             4
MEDEU              4
 Medeu             4
Турксибский        3
Zhetysuskiy        3
 Almaly            3
Medeuskiy          3
Бостандыкский      3
NAURYZBAY          3
ALATAU             3
 Nauryzbay         2
zhetysu            

### Q3 — Validation Specification

| Column | Expected type | Validation rule |
|---|---|---|
| `listing_id` | string | Must be a valid listing identifier; one unique listing should remain after repost resolution |
| `listed_date` | datetime | Must be a valid date |
| `district` | category | Must map to one of the 8 canonical Almaty districts |
| `rooms` | numeric | Must be a positive integer; `studio` will be mapped to 1 |
| `area_m2` | numeric | Must be positive |
| `floor` | numeric | Must be positive and must not exceed `total_floors` |
| `total_floors` | numeric | Must be positive |
| `year_built` | numeric | Must represent a plausible construction year; sentinel values such as 0 are invalid |
| `building_type` | category | Must be a valid building type |
| `ceiling_height_m` | numeric | Must be positive; sentinel values such as 0 are invalid |
| `price_kzt` | numeric | Must be positive |

**Cross-field rule:** `floor <= total_floors`.

In [13]:
cleaning_log = []

before = len(df)

duplicate_count = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

after = len(df)

cleaning_log.append({
    "operation": "Remove exact duplicates",
    "rows_before": before,
    "rows_after": after,
    "cells_changed": 0
})

print("Exact duplicates removed:", duplicate_count)
print("Rows before:", before)
print("Rows after:", after)

Exact duplicates removed: 33
Rows before: 1156
Rows after: 1123


In [14]:
df["listed_date"] = pd.to_datetime(df["listed_date"], errors="coerce")

print("Invalid dates:", df["listed_date"].isna().sum())

Invalid dates: 0


In [15]:
before = len(df)

df = (
    df.sort_values("listed_date")
      .drop_duplicates(subset="listing_id", keep="last")
      .reset_index(drop=True)
)

after = len(df)

cleaning_log.append({
    "operation": "Resolve reposts — keep latest listed_date",
    "rows_before": before,
    "rows_after": after,
    "cells_changed": 0
})

print("Repost rows removed:", before - after)
print("Rows before:", before)
print("Rows after:", after)
print("Duplicate listing IDs remaining:", df["listing_id"].duplicated().sum())

Repost rows removed: 43
Rows before: 1123
Rows after: 1080
Duplicate listing IDs remaining: 0


### Repost strategy

The source file is shuffled, so row order cannot be used to decide which repost to keep. I converted `listed_date` to datetime, sorted the records by date, and kept the most recent record for each `listing_id`. This gives one row per unique listing and retains the latest available version of a reposted advertisement.

In [16]:
district_values = sorted(df["district"].dropna().unique())

print("Number of unique district values:", len(district_values))
print(district_values)

Number of unique district values: 48
[' Alatau ', ' Almaly ', ' Auezov ', ' Bostandyk ', ' Medeu ', ' Nauryzbay ', ' Turksib ', ' Zhetysu ', 'ALATAU', 'ALMALY', 'AUEZOV', 'Alatau', 'Alatauskiy', 'Almalinskiy', 'Almaly', 'Auezov', 'Auezovskiy', 'BOSTANDYK', 'Bostandyk', 'Bostandykskiy', 'MEDEU', 'Medeu', 'Medeuskiy', 'NAURYZBAY', 'Nauryzbay', 'Nauryzbayskiy', 'TURKSIB', 'Turksib', 'Turksibskiy', 'ZHETYSU', 'Zhetysu', 'Zhetysuskiy', 'alatau', 'almaly', 'auezov', 'bostandyk', 'medeu', 'nauryzbay', 'turksib', 'zhetysu', 'Алатауский', 'Алмалинский', 'Ауэзовский', 'Бостандыкский', 'Жетысуский', 'Медеуский', 'Наурызбайский', 'Турксибский']


In [ ]:
before_district = df["district"].copy()

district_clean = df["district"].str.strip().str.lower()

district_mapping = {
   
    "alatau": "Alatau",
    "alatauskiy": "Alatau",
    "алатауский": "Alatau",

   
    "almaly": "Almaly",
    "almalinskiy": "Almaly",
    "алмалинский": "Almaly",

    
    "auezov": "Auezov",
    "auezovskiy": "Auezov",
    "ауэзовский": "Auezov",

 
    "bostandyk": "Bostandyk",
    "bostandykskiy": "Bostandyk",
    "бостандыкский": "Bostandyk",

   
    "medeu": "Medeu",
    "medeuskiy": "Medeu",
    "медеуский": "Medeu",

    
    "nauryzbay": "Nauryzbay",
    "nauryzbayskiy": "Nauryzbay",
    "наурызбайский": "Nauryzbay",

   
    "turksib": "Turksib",
    "turksibskiy": "Turksib",
    "турксибский": "Turksib",

    
    "zhetysu": "Zhetysu",
    "zhetysuskiy": "Zhetysu",
    "жетысуский": "Zhetysu"
}

df["district"] = district_clean.map(district_mapping)

cells_changed = (before_district != df["district"]).sum()

cleaning_log.append({
    "operation": "Standardize district names",
    "rows_before": len(df),
    "rows_after": len(df),
    "cells_changed": cells_changed
})

print("Cells changed:", cells_changed)
print("Unique districts:", df["district"].nunique())
print("Missing districts:", df["district"].isna().sum())
print(df["district"].value_counts())

Cells changed: 166
Unique districts: 8
Missing districts: 0
district
Medeu        161
Turksib      146
Bostandyk    145
Auezov       144
Almaly       127
Zhetysu      126
Nauryzbay    120
Alatau       111
Name: count, dtype: int64


In [19]:
before_rooms = df["rooms"].copy()
before_area = df["area_m2"].copy()
before_price = df["price_kzt"].copy()

df["rooms"] = (
    df["rooms"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace("studio", "1")
)

df["rooms"] = pd.to_numeric(df["rooms"], errors="coerce")

df["area_m2"] = (
    df["area_m2"]
    .astype(str)
    .str.replace("m²", "", regex=False)
    .str.strip()
    .str.replace(",", ".", regex=False)
)

df["area_m2"] = pd.to_numeric(df["area_m2"], errors="coerce")


df["price_kzt"] = (
    df["price_kzt"]
    .astype(str)
    .str.replace("₸", "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["price_kzt"] = pd.to_numeric(df["price_kzt"], errors="coerce")


print("rooms dtype:", df["rooms"].dtype)
print("area_m2 dtype:", df["area_m2"].dtype)
print("price_kzt dtype:", df["price_kzt"].dtype)

print("\nMissing after conversion:")
print("rooms:", df["rooms"].isna().sum())
print("area_m2:", df["area_m2"].isna().sum())
print("price_kzt:", df["price_kzt"].isna().sum())

rooms dtype: int64
area_m2 dtype: float64
price_kzt dtype: int64

Missing after conversion:
rooms: 0
area_m2: 0
price_kzt: 0


In [20]:
rooms_changed = (before_rooms.astype(str) != df["rooms"].astype(str)).sum()
area_changed = (before_area.astype(str) != df["area_m2"].astype(str)).sum()
price_changed = (before_price.astype(str) != df["price_kzt"].astype(str)).sum()

cleaning_log.append({
    "operation": "Parse rooms to numeric",
    "rows_before": len(df),
    "rows_after": len(df),
    "cells_changed": rooms_changed
})

cleaning_log.append({
    "operation": "Parse area_m2 to numeric",
    "rows_before": len(df),
    "rows_after": len(df),
    "cells_changed": area_changed
})

cleaning_log.append({
    "operation": "Parse price_kzt to numeric",
    "rows_before": len(df),
    "rows_after": len(df),
    "cells_changed": price_changed
})

print("Rooms cells changed:", rooms_changed)
print("Area cells changed:", area_changed)
print("Price cells changed:", price_changed)

Rooms cells changed: 28
Area cells changed: 87
Price cells changed: 89


In [21]:
pd.DataFrame(cleaning_log)

,operation,rows_before,rows_after,cells_changed
0,Remove exact duplicates,1156,1123,0
1,Resolve reposts — keep latest listed_date,1123,1080,0
2,Standardize district names,1080,1080,166
3,Parse rooms to numeric,1080,1080,28
4,Parse area_m2 to numeric,1080,1080,87
5,Parse price_kzt to numeric,1080,1080,89


In [22]:
print(df.dtypes)
print("\nRows:", len(df))
print("Unique districts:", df["district"].nunique())
print("Missing districts:", df["district"].isna().sum())
print("Duplicate listing IDs:", df["listing_id"].duplicated().sum())

listing_id                  object
listed_date         datetime64[ns]
district                    object
rooms                        int64
area_m2                    float64
floor                        int64
total_floors                 int64
year_built                 float64
building_type               object
ceiling_height_m           float64
price_kzt                    int64
dtype: object

Rows: 1080
Unique districts: 8
Missing districts: 0
Duplicate listing IDs: 0
